# 🛡️ Phishing Detector — Exploration Notebook

Use this notebook to explore the dataset, features, and model predictions interactively.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from feature_extraction import extract_features, FEATURE_COLUMNS
from predict import PhishingPredictor

sns.set_theme(style='whitegrid')
print('Imports OK')

## 1. Load dataset

In [ ]:
df = pd.read_csv('../data/phishing_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Label counts:\n{df["label"].value_counts()}')
df.head()

## 2. Feature distributions

In [ ]:
numeric_cols = ['url_length','domain_length','num_dots','num_subdomains',
                'domain_entropy','url_entropy','digit_ratio','path_depth']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, numeric_cols):
    for lbl, color in [(0,'#2563eb'),(1,'#dc2626')]:
        subset = df[df['label']==lbl][col]
        ax.hist(subset, bins=30, alpha=0.55, color=color,
                label='Legitimate' if lbl==0 else 'Phishing', density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Feature distributions by class', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlation heatmap

In [ ]:
corr = df[FEATURE_COLUMNS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, annot=False, ax=ax,
            linewidths=0.3)
ax.set_title('Feature correlation matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Real-time predictions

In [ ]:
p = PhishingPredictor(model_dir='../models')

test_urls = [
    'https://www.google.com',
    'https://github.com/login',
    'http://192.168.0.1/verify-account.php?token=%41%42',
    'http://paypal-secure-login.xyz/signin?sessid=abc&redirect=%2F',
    'https://www.amazon.com/dp/B08N5WRWNW',
    'http://secure-ebay-signin.tk/signin/?user=victim',
]

results = p.predict_batch(test_urls)
summary = pd.DataFrame([{
    'URL':         r['url'][:60]+'…' if len(r['url'])>60 else r['url'],
    'Label':       r['label'],
    'Risk':        r['risk_level'],
    'P(phishing)': f"{r['prob_phishing']:.1%}",
} for r in results])
summary

## 5. Feature importance (from saved model)

In [ ]:
import joblib, json

clf    = joblib.load('../models/best_model.pkl')
meta   = json.load(open('../models/metadata.json'))
feats  = meta['feature_cols']

if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
    idx = np.argsort(imp)[::-1][:20]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([feats[i] for i in idx][::-1], imp[idx][::-1], color='#2563eb')
    ax.set_xlabel('Importance')
    ax.set_title('Top-20 feature importances')
    plt.tight_layout()
    plt.show()
elif hasattr(clf, 'coef_'):
    coef = np.abs(clf.coef_[0])
    idx  = np.argsort(coef)[::-1][:20]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([feats[i] for i in idx][::-1], coef[idx][::-1], color='#7c3aed')
    ax.set_xlabel('|Coefficient|')
    ax.set_title('Top-20 feature coefficients (Logistic Regression)')
    plt.tight_layout()
    plt.show()